# Gold v2 — vision expert 히트맵 점검 (LVLM 없음, CPU)

LVLM에 heatmap을 줬을 때 AnomalyCLIP의 Q1 F1이 가장 높았지만(0.675), recall과 FDR도 가장 높았습니다.
이 노트북은 LVLM 없이 **expert 자체**가 결함을 잘 찾는지 봅니다. 새 추론은 없고, Vision Expert full 실행 때 Drive에 저장된 anomaly map(.npy)과 heatmap 이미지를 다시 읽기만 합니다.

- `detection.csv`: expert 점수로 정상/이상을 얼마나 가르는지 (AUROC, 246장)
- `localization.csv`: 녹 20장(정답 박스 있음)에서 heatmap 최고점·ROI 중심이 정답 박스 안에 들어가는 비율, ROI IoU, 박스 안 열 집중도.
  비교용으로 "그냥 이미지 가운데를 찍기"도 같이 계산 (20장 정답이 전부 가운데라서)
- `heat_on_normals.csv`: 정상 이미지에서도 heatmap이 얼마나 뜨거워 보이는지 (map을 이미지마다 0~1로 늘리기 때문)
- `grid_*.png`: 원본+정답 박스 옆에 expert 4종 heatmap을 나란히 (빨강 = 정답 박스, 노랑 = expert ROI, 하늘색 십자 = 최고점)

런타임: **CPU로 충분** (GPU 불필요). Drive에서 파일 2천여 개를 읽어서 5~15분 정도.
결과: `MyDrive/lvlm_insplad_results/gold_v2_expert_heatmap_check/` → GitHub `results/latest/gold_v2_expert_heatmap_check/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
VX_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_expert_heatmap_check"
EXPERTS = ["patchcore","anomalyclip","dinomaly","reconpatch_lite"]
PUBLISH = True

SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
MERGED_ROOT = DRIVE_ROOT / VX_FOLDER / "merged_evidence"
MERGED_MANIFEST = MERGED_ROOT / "roi_manifest_all.csv"
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, *, env=None, check=True):
    # Colab은 subprocess 출력을 숨기므로 한 줄씩 바로 보여줌
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, cmd)
    return code

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음). 경로를 알면 설정 셀의 SUP_ZIP_PATH에 넣으세요.", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
print("SUP:", SUP_ZIP)
missing = [str(p) for p in (MERGED_MANIFEST,) if not p.is_file()]
if not sorted(QA_DIR.glob("master_qa_*.jsonl")):
    missing.append(str(QA_DIR / "master_qa_*.jsonl"))
if not sorted(QA_DIR.glob("localization_qa_*.jsonl")):
    missing.append(str(QA_DIR / "localization_qa_*.jsonl"))
if missing:
    raise FileNotFoundError("입력 없음 (Vision Expert full 노트북의 통합 단계가 끝났는지 확인): " + ", ".join(missing))
print("통합 manifest:", MERGED_MANIFEST)

In [ ]:
# 실행 — 다시 실행해도 같은 결과를 덮어씀 (5~15분)
run([sys.executable, REPO_DIR / "scripts/build_gold_v2_expert_heatmap_check.py",
     "--qa-dir", QA_DIR, "--manifest", MERGED_MANIFEST, "--artifact-root", MERGED_ROOT,
     "--sup-zip", SUP_ZIP, "--out-dir", RUN_ROOT, "--experts", *EXPERTS])

In [ ]:
# 결과 보기
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 220)
print("[1] 탐지: expert 점수의 AUROC (0.5 = 찍기, 1.0 = 완벽)")
display(pd.read_csv(RUN_ROOT / "detection.csv"))
print("\n[2] 위치: 녹 20장, 정답 박스 안에 들어간 비율 (image_centre = 그냥 가운데 찍기)")
display(pd.read_csv(RUN_ROOT / "localization.csv"))
print("\n[3] 정상 이미지에서도 뜨거운 면적")
display(pd.read_csv(RUN_ROOT / "heat_on_normals.csv"))
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
print(json.dumps(status, ensure_ascii=False, indent=1))
for name in status["grids"]:
    print("\n" + name)
    display(Image(filename=str(RUN_ROOT / name), width=1300))

In [ ]:
if PUBLISH:
    status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
    files = ["detection.csv", "localization.csv", "localization_by_image.csv", "heat_on_normals.csv",
             "run_status.json", "run_metadata.json", *status["grids"]]
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", RUN_ROOT,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_expert_heatmap_check",
         "--files", *[f for f in files if (RUN_ROOT / f).is_file()]], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_expert_heatmap_check/")

## 읽는 법

- **AUROC**가 0.5 근처면 그 expert 점수는 정상/이상을 거의 못 가름. `auroc_asset_macro`는 자산별로 따로 잰 뒤 평균 (자산마다 점수 크기가 달라서 이게 더 공정함)
- **peak_in_gt_rate**가 `image_centre`(그냥 가운데 찍기)보다 낮으면, 그 expert는 위치를 찍기보다 못 맞힘
- **mean_heat_in_box_ratio** > 1이면 열이 정답 박스에 몰려 있음, 1 근처면 이미지 전체에 퍼져 있음
- `grid_normal.png`: 정상 이미지에도 뜨거운 점이 보이면, LVLM이 heatmap을 보고 "이상"이라고 더 자주 답한 이유가 됨
- 정답 박스가 있는 이미지는 20장(전부 lightning-rod-suspension 녹)뿐이라 위치 결과는 참고용